# Pseudobulk-Conditioned BRCA2021 Dataset And Per-Sample UMAP

This notebook provides two views:

1. A global BRCA2021-wide real-vs-generated comparison using pooled pseudobulk-conditioned generated samples.
2. A per-`SampleID` comparison where each real BRCA2021 sample is plotted together with its corresponding generated sample in the same UMAP, colored by `real` vs `generated`.


In [ ]:
from pathlib import Path
import sys

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
import scanpy as sc

project_root = Path.cwd()
if not (project_root / 'guided_diffusion').exists() and (project_root.parent / 'guided_diffusion').exists():
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from guided_diffusion.cell_datasets_loader import (
    decode_latents_with_vae,
    encode_cells_with_vae,
    read_preprocessed_adata,
)

In [ ]:
# Main configuration.
data_dir = '<path/to/data.h5ad>'
vae_path = str(project_root / 'output' / 'checkpoint' / 'AE' / 'brca2021_VAE_alltrain' / 'model_seed=1234_step=199999.pt')
generated_glob = str(project_root / 'output' / 'simulated_samples' / 'brca2021_pseudobulk_1M_alltrain_*.npz')

sample_key = 'SampleID'
celltype_key = 'curated_celltypes'

# Caps used to keep UMAPs readable and notebook runtime manageable.
max_global_real_per_sample = 800
max_global_generated_per_sample = 800
max_cells_per_sample_plot = 2500
random_seed = 1234

generated_glob


In [ ]:
def scalar_from_npz(value):
    value = np.asarray(value)
    if value.shape == ():
        return value.item()
    flat = value.reshape(-1)
    return flat[0].item() if hasattr(flat[0], 'item') else flat[0]


def compute_proxy_pseudobulk(cell_matrix):
    cell_matrix = np.asarray(cell_matrix, dtype=np.float32)
    linear = np.expm1(cell_matrix)
    np.maximum(linear, 0.0, out=linear)
    # Sanitize the result to prevent overflow issues downstream
    linear = np.nan_to_num(linear, nan=0.0, posinf=0.0, neginf=0.0)
    return np.log1p(linear.mean(axis=0, dtype=np.float64)).astype(np.float32)


def subsample_array(matrix, max_cells, seed=1234):
    if matrix.shape[0] <= max_cells:
        return matrix, np.arange(matrix.shape[0])
    rng = np.random.default_rng(seed)
    idx = np.sort(rng.choice(matrix.shape[0], size=max_cells, replace=False))
    return matrix[idx], idx


def build_umap_adata(matrix, obs, gene_names=None, n_neighbors=10, n_pcs=20):
    # Sanitize matrix immediately to prevent overflow issues
    mat = np.asarray(matrix, dtype=np.float32)
    n_inf = np.isinf(mat).sum()
    n_nan = np.isnan(mat).sum()
    if n_inf > 0 or n_nan > 0:
        print(f'[build_umap_adata] sanitizing matrix: {n_inf} inf, {n_nan} nan values replaced with 0')
        mat = np.nan_to_num(mat, nan=0.0, posinf=0.0, neginf=0.0)
    
    # Clip extreme values to prevent numerical instability
    mat = np.clip(mat, -100, 100)
    
    adata_plot = ad.AnnData(mat)
    if gene_names is not None:
        adata_plot.var_names = [str(g) for g in gene_names]
    adata_plot.obs = obs.copy()
    
    # Use cell_ranger flavor which is more numerically stable
    try:
        sc.pp.highly_variable_genes(adata_plot, flavor='cell_ranger', n_top_genes=2000)
    except Exception as e:
        print(f'[build_umap_adata] HVG selection failed ({e}), using all genes instead')
        adata_plot.var['highly_variable'] = True
    
    adata_plot.raw = adata_plot
    adata_plot = adata_plot[:, adata_plot.var.highly_variable].copy()
    sc.pp.scale(adata_plot, max_value=10)
    sc.tl.pca(adata_plot, svd_solver='arpack')
    sc.pp.neighbors(adata_plot, n_neighbors=n_neighbors, n_pcs=min(n_pcs, adata_plot.obsm['X_pca'].shape[1]))
    sc.tl.umap(adata_plot)
    return adata_plot


def discover_sample_files(glob_pattern):
    paths = sorted(Path().glob(glob_pattern) if not Path(glob_pattern).is_absolute() else Path(glob_pattern).parent.glob(Path(glob_pattern).name))
    return paths


In [ ]:
def build_umap_ingest(real_matrix, gen_matrix, obs_real, obs_gen, n_neighbors=10, n_pcs=20):
    """Fit UMAP on real cells only; project generated cells in via sc.tl.ingest."""
    n_genes = real_matrix.shape[1]
    var_names = [str(i) for i in range(n_genes)]

    mat_real = np.nan_to_num(np.clip(real_matrix.astype(np.float32), -100, 100))
    mat_gen  = np.nan_to_num(np.clip(gen_matrix.astype(np.float32),  -100, 100))

    # HVG selection on real cells only
    adata_full = ad.AnnData(mat_real.copy())
    adata_full.var_names = var_names
    try:
        sc.pp.highly_variable_genes(adata_full, min_mean=0.0125, max_mean=3, min_disp=0.5)
    except Exception:
        adata_full.var['highly_variable'] = True
    hvg_mask      = adata_full.var['highly_variable'].values
    hvg_var_names = [var_names[i] for i, h in enumerate(hvg_mask) if h]

    # Scaling parameters derived from real cells only
    real_hvg  = mat_real[:, hvg_mask]
    gene_mean = real_hvg.mean(axis=0)
    gene_std  = real_hvg.std(axis=0)
    gene_std[gene_std < 1e-8] = 1.0

    real_scaled = np.clip((real_hvg - gene_mean) / gene_std, -10, 10)
    gen_scaled  = np.clip((mat_gen[:, hvg_mask] - gene_mean) / gene_std, -10, 10)

    # Reference UMAP fitted on real cells
    adata_ref = ad.AnnData(real_scaled)
    adata_ref.var_names = hvg_var_names
    adata_ref.obs = obs_real.copy()
    sc.tl.pca(adata_ref, svd_solver='arpack')
    sc.pp.neighbors(adata_ref, n_neighbors=n_neighbors,
                    n_pcs=min(n_pcs, adata_ref.obsm['X_pca'].shape[1]))
    sc.tl.umap(adata_ref)

    # Query: generated cells with same scaling applied
    adata_gen = ad.AnnData(gen_scaled)
    adata_gen.var_names = hvg_var_names
    adata_gen.obs = obs_gen.copy()
    sc.tl.ingest(adata_gen, adata_ref, embedding_method='umap')

    return ad.concat([adata_ref, adata_gen], uns_merge='first')


def build_umap_latent(real_matrix, gen_latents, obs_real, obs_gen, vae_path, n_neighbors=10):
    """Joint UMAP in 128-dim VAE latent space: encoder(real) vs diffusion(generated)."""
    real_latents = encode_cells_with_vae(real_matrix, vae_path, hidden_dim=128)

    combined = np.concatenate([real_latents, gen_latents], axis=0)
    obs = pd.concat([obs_real, obs_gen], ignore_index=True)

    adata = ad.AnnData(combined)
    adata.obs = obs
    # latent space is 128-dim: use X directly, skip PCA
    sc.pp.neighbors(adata, use_rep='X', n_neighbors=n_neighbors)
    sc.tl.umap(adata)
    return adata

In [ ]:
generated_paths = discover_sample_files(generated_glob)
len(generated_paths), generated_paths[:5]


In [ ]:
adata = read_preprocessed_adata(data_dir)
gene_names = np.asarray(adata.var_names).astype(str)
print('real BRCA2021 cells:', adata.shape)


In [ ]:
real_sample_ids = sorted(adata.obs[sample_key].astype(str).unique())
generated_sample_ids = []
for path in generated_paths:
    npzfile = np.load(path, allow_pickle=True)
    generated_sample_ids.append(str(scalar_from_npz(npzfile['source_sample_id'])))
generated_sample_ids = sorted(set(generated_sample_ids))
missing_sample_ids = [sample_id for sample_id in real_sample_ids if sample_id not in generated_sample_ids]

print('real SampleIDs in BRCA2021:', len(real_sample_ids))
print('generated SampleIDs found on disk:', len(generated_sample_ids))
print('generated SampleIDs:', generated_sample_ids[:10])
print('missing SampleIDs:', missing_sample_ids[:10])
if len(generated_sample_ids) < len(real_sample_ids):
    print('\nThe notebook only compares SampleIDs that already have generated .npz files.')
    print('If just one SampleID appears here, only one generated sample has been created so far.')


The per-sample UMAP loop below runs over `generated_by_sample`, which is built from the discovered `brca2021_pseudobulk_1M_*.npz` files.

So if there is only one generated file in `output/simulated_samples/`, you will only see one sample overlay in this notebook.

In [ ]:
generated_by_sample = {}
summary_rows = []

for path in generated_paths:
    stem = path.stem
    if '_test_' in stem:
        split = 'test'
    elif '_train_' in stem:
        split = 'train'
    else:
        split = 'unknown'

    npzfile = np.load(path, allow_pickle=True)
    sample_id = str(scalar_from_npz(npzfile['source_sample_id']))
    conditioning_reduction = str(scalar_from_npz(npzfile['pseudobulk_reduction']))
    input_pseudobulk = np.asarray(npzfile['input_pseudobulk'], dtype=np.float32)
    generated_latents = np.asarray(npzfile['cell_gen'], dtype=np.float32)
    generated_cells = decode_latents_with_vae(generated_latents, vae_path, hidden_dim=128)

    sample_mask = adata.obs[sample_key].astype(str).to_numpy() == sample_id
    real_adata = adata[sample_mask].copy()
    if real_adata.n_obs == 0:
        print(f'skipping {sample_id}: not found in real dataset after preprocessing')
        continue

    real_cells = real_adata.X.toarray() if hasattr(real_adata.X, 'toarray') else np.asarray(real_adata.X)
    real_cells = np.asarray(real_cells, dtype=np.float32)
    real_celltypes = real_adata.obs[celltype_key].astype(str).to_numpy() if celltype_key in real_adata.obs else None
    real_celltype_major = real_adata.obs['celltype_major'].astype(str).to_numpy() if 'celltype_major' in real_adata.obs else None
    real_celltype_minor = real_adata.obs['celltype_minor'].astype(str).to_numpy() if 'celltype_minor' in real_adata.obs else None

    generated_proxy_pseudobulk = compute_proxy_pseudobulk(generated_cells)
    real_proxy_pseudobulk = compute_proxy_pseudobulk(real_cells)

    generated_by_sample[sample_id] = {
        'path': path,
        'split': split,
        'conditioning_reduction': conditioning_reduction,
        'generated_latents': generated_latents,
        'generated_cells': generated_cells,
        'real_cells': real_cells,
        'real_celltypes': real_celltypes,
        'real_celltype_major': real_celltype_major,
        'real_celltype_minor': real_celltype_minor,
        'input_pseudobulk': input_pseudobulk,
        'generated_proxy_pseudobulk': generated_proxy_pseudobulk,
        'real_proxy_pseudobulk': real_proxy_pseudobulk,
    }

    summary_rows.append({
        'sample_id': sample_id,
        'split': split,
        'num_real_cells': real_cells.shape[0],
        'num_generated_cells': generated_cells.shape[0],
        'conditioning_vs_generated_proxy_pearson': np.corrcoef(input_pseudobulk, generated_proxy_pseudobulk)[0, 1],
        'conditioning_vs_generated_proxy_spearman': stats.spearmanr(input_pseudobulk, generated_proxy_pseudobulk).correlation,
        'real_proxy_vs_generated_proxy_pearson': np.corrcoef(real_proxy_pseudobulk, generated_proxy_pseudobulk)[0, 1],
        'path': str(path.relative_to(project_root)),
    })

summary_df = pd.DataFrame(summary_rows).sort_values('sample_id').reset_index(drop=True)
summary_df

## Global Real Vs Generated BRCA2021 View

This is the closest equivalent to the old real-vs-generated dataset-level UMAP. Since the model is now conditioned, we pool generated cells from all available sample-specific pseudobulk conditions.

In [ ]:
global_blocks = []
global_obs_parts = []

for sample_id, payload in generated_by_sample.items():
    real_sub, real_idx = subsample_array(payload['real_cells'], max_global_real_per_sample, seed=random_seed)
    gen_sub, _ = subsample_array(payload['generated_cells'], max_global_generated_per_sample, seed=random_seed + 1)

    if payload['real_celltype_major'] is not None:
        celltype_major_sub = list(payload['real_celltype_major'][real_idx])
    else:
        celltype_major_sub = ['NA'] * real_sub.shape[0]
    if payload['real_celltype_minor'] is not None:
        celltype_minor_sub = list(payload['real_celltype_minor'][real_idx])
    else:
        celltype_minor_sub = ['NA'] * real_sub.shape[0]

    global_blocks.append(real_sub)
    global_blocks.append(gen_sub)

    global_obs_parts.append(pd.DataFrame({
        'source': ['real'] * real_sub.shape[0] + ['generated'] * gen_sub.shape[0],
        'sample_id': sample_id,
        'split': payload['split'],
        'celltype_major': celltype_major_sub + ['generated'] * gen_sub.shape[0],
        'celltype_minor': celltype_minor_sub + ['generated'] * gen_sub.shape[0],
    }))

global_matrix = np.concatenate(global_blocks, axis=0)
global_obs = pd.concat(global_obs_parts, ignore_index=True)
global_obs['source'] = pd.Categorical(global_obs['source'])
global_obs['sample_id'] = pd.Categorical(global_obs['sample_id'])
global_obs['split'] = pd.Categorical(global_obs['split'])
global_obs['celltype_major'] = pd.Categorical(global_obs['celltype_major'])
global_obs['celltype_minor'] = pd.Categorical(global_obs['celltype_minor'])

global_matrix.shape, global_obs.shape

In [ ]:
adata_global = build_umap_adata(global_matrix, global_obs, gene_names=gene_names)
adata_global

In [ ]:
# Cache this embedding to disk so other notebooks (e.g. the marker-gene UMAP audits)
# can load the exact same coordinates instead of recomputing PCA/UMAP themselves --
# arpack/UMAP are not guaranteed bit-reproducible across separate process runs even
# with the same random_state, so recomputing would not give a pixel-identical layout.
import os
os.environ['HDF5_USE_FILE_LOCKING'] = 'FALSE'  # avoid h5py lock errors on NFS-mounted home dirs

CACHE_DIR = project_root / 'output' / 'cached_umap'
CACHE_DIR.mkdir(parents=True, exist_ok=True)
CACHE_PATH = CACHE_DIR / 'brca2021_global_real_vs_generated_umap_alltrain.h5ad'
adata_global.write_h5ad(CACHE_PATH)
print(f'Saved global UMAP embedding to {CACHE_PATH}')

In [ ]:
print(adata_global.obs['celltype_major'].value_counts())
print()
print(adata_global.obs['celltype_minor'].value_counts())

In [ ]:
sc.pl.umap(
    adata_global,
    color='source',
    size=8,
    title='BRCA pooled real vs pooled pseudobulk-conditioned generated cells (train + test)',
)

In [ ]:
sc.pl.umap(
    adata_global,
    color='celltype_major',
    size=8,
    title='BRCA pooled real and generated cells colored by SampleID (train + test)',
)

The same pooled embedding (`adata_global`), reused as-is. Each panel below still shows every cell for full context, but only one split (train or test) is colored by `source`; the other split is grayed into the background. This makes it possible to check whether real cells that appear unmatched by generated cells in the plot above are disproportionately from the test split.

In [ ]:
highlight_palette = {'real': '#1f77b4', 'generated': '#ff7f0e'}

for focus_split in ['train', 'test']:
    other_split = 'test' if focus_split == 'train' else 'train'
    is_focus = (adata_global.obs['split'] == focus_split).to_numpy()

    other_label = f'other ({other_split})'
    highlight = adata_global.obs['source'].astype(str).copy()
    highlight[~is_focus] = other_label
    adata_global.obs[f'highlight_{focus_split}'] = pd.Categorical(
        highlight, categories=['real', 'generated', other_label]
    )

    # Draw background cells first so the highlighted split renders on top.
    draw_order = np.argsort(is_focus)
    palette = {**highlight_palette, other_label: 'lightgray'}

    sc.pl.umap(
        adata_global[draw_order],
        color=f'highlight_{focus_split}',
        palette=palette,
        size=8,
        title=f'BRCA global UMAP: real vs generated {focus_split} cells ({other_split} cells in gray)',
    )

## Per-Sample Proxy Pseudobulk Agreement Summary

The summary below uses decoded-cell proxy pseudobulks computed as `log1p(mean(expm1(cell_matrix), axis=0))` for the real and generated cells. This avoids applying the training-time `sum` reduction directly to decoded cell-space values, which is no longer comparable to the stored conditioning pseudobulk after the library-normalization change.


In [ ]:
summary_df[['sample_id', 'num_real_cells', 'num_generated_cells', 'conditioning_vs_generated_proxy_pearson', 'conditioning_vs_generated_proxy_spearman', 'real_proxy_vs_generated_proxy_pearson']]


In [ ]:
split_colors = {'train': '#1f77b4', 'test': '#d62728'}

plt.figure(figsize=(8, 5))
plot_df = summary_df.sort_values('conditioning_vs_generated_proxy_pearson')
bar_colors = plot_df['split'].map(split_colors)
plt.barh(plot_df['sample_id'], plot_df['conditioning_vs_generated_proxy_pearson'], color=bar_colors)
plt.xlabel('pearson correlation: conditioning pseudobulk vs generated proxy pseudobulk')
plt.ylabel('SampleID',fontsize=7)
plt.title('Per-sample conditioning vs generated proxy agreement')
# legend_handles = [plt.Rectangle((0, 0), 1, 1, color=color) for color in split_colors.values()]
# plt.legend(legend_handles, split_colors.keys(), title='split', loc='lower right')
legend_handles = [plt.Rectangle((0, 0), 1, 1, color=color) for color in split_colors.values()]
plt.legend(
    legend_handles, split_colors.keys(),
    title='split',
    loc='lower right',
    fontsize=6,           
    title_fontsize=6,     
    handlelength=2,     
    handleheight=2,     
    borderpad=0.6,        
    labelspacing=0.6,     
    handletextpad=0.6     
)
plt.tight_layout()
plt.show()


## Per-Sample UMAP: Seurat Dispersion-Based HVG

HVG selection uses `min_mean=0.0125, max_mean=3, min_disp=0.5` (Seurat flavor) on the **joint** real+generated pool.
Dispersion is normalized within mean-expression bins, so genes that vary only due to a systematic real-vs-generated offset
do not pass the `min_disp=0.5` threshold and are excluded from PCA — unlike `cell_ranger` top-N which forces exactly
2000 genes regardless of what drives their variance.

In [ ]:
def build_umap_adata_seurat(matrix, obs, n_neighbors=10, n_pcs=20):
    """Build UMAP using Seurat dispersion-based HVG on the joint real+generated pool."""
    mat = np.asarray(matrix, dtype=np.float32)
    mat = np.nan_to_num(mat, nan=0.0, posinf=0.0, neginf=0.0)
    adata_plot = ad.AnnData(mat)
    adata_plot.obs = obs.copy()
    sc.pp.highly_variable_genes(adata_plot, min_mean=0.0125, max_mean=3, min_disp=0.5)
    adata_plot.raw = adata_plot
    adata_plot = adata_plot[:, adata_plot.var.highly_variable].copy()
    sc.pp.scale(adata_plot)
    sc.tl.pca(adata_plot, svd_solver='arpack')
    sc.pp.neighbors(adata_plot, n_neighbors=n_neighbors, n_pcs=min(n_pcs, adata_plot.obsm['X_pca'].shape[1]))
    sc.tl.umap(adata_plot)
    return adata_plot

In [ ]:
for sample_id in sorted(generated_by_sample):
    payload = generated_by_sample[sample_id]
    split = payload['split']

    real_sub, _ = subsample_array(payload['real_cells'],      max_cells_per_sample_plot, seed=random_seed)
    gen_sub,  _ = subsample_array(payload['generated_cells'], max_cells_per_sample_plot, seed=random_seed + 1)

    combined = np.concatenate((real_sub, gen_sub), axis=0)
    obs = pd.DataFrame({
        'source': pd.Categorical(['real'] * real_sub.shape[0] + ['generated'] * gen_sub.shape[0]),
    })

    adata_sample = build_umap_adata_seurat(combined, obs)
    n_hvg = adata_sample.n_vars
    title = f'{sample_id} [{split.upper()}]: real vs generated  (Seurat HVG, n={n_hvg})'
    sc.pl.umap(adata_sample, color='source', size=10, title=title)

## Per-Sample UMAP: VAE Latent Space (Joint Embedding)

Real cells are encoded by the VAE encoder; generated latents come directly from the diffusion model.
UMAP is run jointly on the combined 128-dim latent pool — both groups compete equally in the neighbor graph.
This is the most direct diagnostic: if the diffusion model is faithful, real and generated latents overlap here.
If they overlap here but not in the ingest UMAP above, the issue is in the decoder, not the diffusion model.

In [ ]:
for sample_id in sorted(generated_by_sample):
    payload = generated_by_sample[sample_id]
    split = payload['split']
    real_sub, _   = subsample_array(payload['real_cells'],       max_cells_per_sample_plot, seed=random_seed)
    gen_lat_sub, _ = subsample_array(payload['generated_latents'], max_cells_per_sample_plot, seed=random_seed + 1)

    obs_real = pd.DataFrame({'source': pd.Categorical(['real']      * real_sub.shape[0])})
    obs_gen  = pd.DataFrame({'source': pd.Categorical(['generated'] * gen_lat_sub.shape[0])})

    adata_sample = build_umap_latent(real_sub, gen_lat_sub, obs_real, obs_gen, vae_path=vae_path)
    title = f'{sample_id} [{split.upper()}] latent: encoder(real) vs diffusion(generated)'
    sc.pl.umap(adata_sample, color='source', size=10, title=title)

## Optional: Inspect One Sample In More Detail

Set `focus_sample_id` below to inspect its conditioning-vs-generated proxy scatter plot and the largest gene-level proxy differences.


In [ ]:
focus_sample_id = summary_df.loc[0, 'sample_id'] if len(summary_df) else None
focus_sample_id


In [ ]:
if focus_sample_id is not None:
    payload = generated_by_sample[focus_sample_id]
    plt.figure(figsize=(6, 6))
    x = payload['input_pseudobulk']
    y = payload['generated_proxy_pseudobulk']
    xy_min = float(min(x.min(), y.min()))
    xy_max = float(max(x.max(), y.max()))
    plt.scatter(x, y, s=6, alpha=0.4)
    plt.plot([xy_min, xy_max], [xy_min, xy_max], color='orange')
    plt.xlabel('conditioning pseudobulk')
    plt.ylabel('generated proxy pseudobulk')
    plt.title(f'{focus_sample_id}: conditioning vs generated proxy pseudobulk')
    plt.show()

    gene_diff = pd.DataFrame({
        'gene': gene_names,
        'conditioning_pseudobulk': payload['input_pseudobulk'],
        'real_proxy_pseudobulk': payload['real_proxy_pseudobulk'],
        'generated_proxy_pseudobulk': payload['generated_proxy_pseudobulk'],
    })
    gene_diff['abs_conditioning_generated_proxy_diff'] = (gene_diff['conditioning_pseudobulk'] - gene_diff['generated_proxy_pseudobulk']).abs()
    display(gene_diff.sort_values('abs_conditioning_generated_proxy_diff', ascending=False).head(20))
